In [18]:
import os
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb

# ==========================================
# 1. LOAD MASTER MODELING MATRIX
# ==========================================
data_path = '../modelData/master_modeling_matrix.csv'
df = pd.read_csv(data_path, index_col='loan_id')
df.index = df.index.astype(str)

cols_to_drop = ['target', 'status', 'is_closed', 'loan_date', 'account_id']
feature_cols = [c for c in df.columns if c not in cols_to_drop]

X = df[feature_cols]
y = df['target']

# ==========================================
# 2. CALCULATE IMBALANCE WEIGHT & FIT WITH HEAVY REGULARIZATION
# ==========================================
neg_count = (y == 0).sum()
pos_count = (y == 1).sum()
scale_pos_weight = neg_count / pos_count

print('Training heavily regularized XGBoost model...')
final_xgb = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=1,             # Keep shallow trees
    learning_rate=0.05,      # Moderate learning rate
    subsample=0.9,           # Slightly more data per tree (90%)
    colsample_bytree=0.9,    # Slightly more features per tree (90%)
    reg_lambda=1.0,          # Mild L2 regularization
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1,
)
final_xgb.fit(X, y)

# ==========================================
# 3. SAVE MODEL, THRESHOLD & FEATURE LIST
# ==========================================
model_artifact = {
    'model': final_xgb,
    'optimal_threshold': 0.2466,
    'feature_cols': feature_cols,
}

output_filename = 'credit_default_xgboost_model.pkl'
joblib.dump(model_artifact, output_filename)

print('=' * 50)
print(f' SUCCESS! Heavily Regularized Model saved as: {output_filename}')
print('=' * 50)

Training heavily regularized XGBoost model...
 SUCCESS! Heavily Regularized Model saved as: credit_default_xgboost_model.pkl


In [19]:
import joblib
import pandas as pd

# Load the saved package
artifact = joblib.load('credit_default_xgboost_model.pkl')
model = artifact['model']
threshold = artifact['optimal_threshold']
features = artifact['feature_cols']

# Assuming 'new_applications_df' is your incoming batch of preprocessed data
# X_new = new_applications_df[features]

# Predict default probabilities
# default_probs = model.predict_proba(X_new)[:, 1]

# Apply your recall-optimized threshold
# final_decisions = (default_probs >= threshold).astype(int)  # 1 = Reject/Review, 0 = Approve